# MrScraper Price Intelligence

This notebook is the report and walkthrough for the price reconstruction pipeline. The implementation lives in `src/`, and the runnable entrypoints live in `scripts/`.

The goal is to fill missing outage-day prices using historical marketplace data plus the available same-day anchor prices.


## How To Run

Full validation:

```bash
.venv/bin/python scripts/run_validation.py
```

Final test prediction:

```bash
.venv/bin/python scripts/predict_test.py
```

Fast smoke checks:

```bash
.venv/bin/python scripts/run_validation.py --sample-rows 5000 --validation-days 1 --anchors-per-day 20 --iterations 5 --output-dir outputs/smoke_validation
.venv/bin/python scripts/predict_test.py --iterations 5 --output-dir outputs/smoke_prediction
```


In [ ]:
from pathlib import Path
from dataclasses import replace
import sys

import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd()))

from src.config import PipelineConfig
from src.data import load_data
from src.features import build_features, get_cat_feature_indices, get_feature_columns
from src.inference import run_prediction
from src.validation import run_outage_validation

config = PipelineConfig()
config


## Data Overview

The train file contains historical prices. The test file contains rows from the outage window, where most `price` values are missing and the non-missing prices act as same-day anchors.


In [13]:
train_raw, test_raw = load_data(config.train_path, config.test_path)

summary = pd.DataFrame(
    [
        {
            "dataset": "train",
            "rows": len(train_raw),
            "columns": train_raw.shape[1],
            "min_capturedAt": train_raw[config.date_col].min(),
            "max_capturedAt": train_raw[config.date_col].max(),
            "missing_price": train_raw[config.target].isna().sum(),
        },
        {
            "dataset": "test",
            "rows": len(test_raw),
            "columns": test_raw.shape[1],
            "min_capturedAt": test_raw[config.date_col].min(),
            "max_capturedAt": test_raw[config.date_col].max(),
            "missing_price": test_raw[config.target].isna().sum(),
        },
    ]
)
summary


,dataset,rows,columns,min_capturedAt,max_capturedAt,missing_price
0,train,306226,26,2025-01-01 22:37:36.424,2025-03-22 04:27:05.302,0
1,test,25900,26,2025-03-22 04:27:05.325,2025-03-24 23:30:04.453,25600


In [14]:
train_raw.head()

,capturedAt,shopId,itemId,modelId,price,priceBeforeDiscount,promotionId,cat_id,stock,normal_stock,...,item_price_max,review_rating,total_rating_count,cmt_count,shop_rating,shop_response_rate,shop_follower_count,is_official_shop,is_verified,is_preferred_plus_seller
0,2025-01-01 22:37:36.424,1008004,29302587134,185168612932,41900000,68000000,534463751667712,100630,NaN,NaN,...,41900000,0.0,0,0,4.975239,55.0,158,f,f,f
1,2025-01-02 01:45:48.634,1006671485,18087284916,166287259494,16900000,0,0,100013,NaN,NaN,...,14900000,5.0,1,1,4.936441,70.0,3955,t,f,f
2,2025-01-02 01:45:48.634,1006671485,18087284916,166287259497,11900000,13900000,474465164066816,100013,NaN,NaN,...,14900000,5.0,1,1,4.936441,70.0,3955,t,f,f
3,2025-01-02 01:45:48.634,1006671485,18087284916,166287259492,16900000,0,0,100013,NaN,NaN,...,14900000,5.0,1,1,4.936441,70.0,3955,t,f,f
4,2025-01-02 01:45:48.634,1006671485,18087284916,166287259490,16900000,0,0,100013,NaN,NaN,...,14900000,5.0,1,1,4.936441,70.0,3955,t,f,f


In [15]:
test_raw.head()

,capturedAt,shopId,itemId,modelId,price,priceBeforeDiscount,promotionId,cat_id,stock,normal_stock,...,item_price_max,review_rating,total_rating_count,cmt_count,shop_rating,shop_response_rate,shop_follower_count,is_official_shop,is_verified,is_preferred_plus_seller
0,2025-03-22 04:27:05.325,1009757562,27904817781,139002028392,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-03-22 04:27:05.325,1009757562,27904817781,139002028392,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2025-03-22 04:27:05.325,1009757562,27904817781,241162534603,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2025-03-22 04:27:05.325,1009757562,27904817781,241162534603,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2025-03-22 04:27:05.325,1009757562,27904817781,241162534604,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Pipeline Design

The pipeline is intentionally split into simple stages:

1. Create normal row-level features.
2. Create leakage-safe historical aggregate features.
3. Train a global CatBoost model on `log1p(price)`.
4. Predict the outage rows in log-price space.
5. Blend the model prediction with an entity historical median prior.
6. Use anchor residuals to calibrate predictions by day.
7. Fill missing prices and preserve known anchor prices.

This structure matches the test setup: anchors are available at prediction time, so they can be used for calibration without leaking hidden prices.


## Feature Engineering

### Row-Level Features

The row-level features come from the current row only:

- time: `hour`, `dayofweek`, `day`, `month`, `is_weekend`
- discount: `has_discount`, `discount_ratio`, `price_before_log`, `raw_discount_log`, `show_discount_ratio`
- stock: `stock_ratio`, `stock_gap`, `is_low_stock`
- engagement: `total_rating_count_log`, `cmt_count_log`, `shop_follower_count_log`, `review_strength`, `shop_strength`
- categorical IDs: `shopId`, `itemId`, `modelId`, `promotionId`, `cat_id`, `brand`
- booleans: free shipping, pre-order, official/verified/preferred seller flags

### Historical Features

For each of `modelId`, `itemId`, `shopId`, `cat_id`, and `brand`, the pipeline computes historical log-price statistics:

- count
- mean
- median
- standard deviation
- minimum
- maximum

It also creates two fallback features:

- recent-price features for `modelId` and `itemId`: last observed price, last 3-observation median, last 7-observation median, prior count, and hours since last seen.
- `fallback_entity_price_log`: most specific available recent/entity prior, falling back from `modelId` last price to `itemId` last price, then historical medians from broader groups.
- `fallback_entity_history_count`: the matching amount of historical evidence.


In [16]:
# Build features on a small chronological sample for inspection only.
# Full training uses the complete dataset through the scripts.
inspect_history = train_raw.sort_values(config.date_col).head(10000)
inspect_features = build_features(inspect_history, inspect_history.head(1000), config)
feature_cols = get_feature_columns(inspect_features, config)
cat_idx = get_cat_feature_indices(inspect_features, feature_cols, config)

pd.DataFrame(
    {
        "n_features": [len(feature_cols)],
        "n_categorical_features": [len(cat_idx)],
    }
)


,n_features,n_categorical_features
0,86,6


In [17]:
historical_feature_preview = [
    col for col in inspect_features.columns
    if col.endswith("_log") or col.endswith("_count") or col.startswith("fallback_entity")
]

pd.Series(historical_feature_preview, name="historical_or_log_feature").head(40)


0                  total_rating_count
1                           cmt_count
2                 shop_follower_count
3                    price_before_log
4                    raw_discount_log
5              total_rating_count_log
6                       cmt_count_log
7             shop_follower_count_log
8          modelId_recent_prior_count
9              modelId_last_price_log
10    modelId_last_3_median_price_log
11    modelId_last_7_median_price_log
12          itemId_recent_prior_count
13              itemId_last_price_log
14     itemId_last_3_median_price_log
15     itemId_last_7_median_price_log
16                modelId_price_count
17             modelId_price_mean_log
18           modelId_price_median_log
19              modelId_price_std_log
20              modelId_price_min_log
21              modelId_price_max_log
22                 itemId_price_count
23              itemId_price_mean_log
24            itemId_price_median_log
25               itemId_price_std_log
26          

## Entity Blending

The model already receives historical features, but the pipeline also applies an explicit shrinkage blend after prediction:

```text
entity_weight = count / (count + entity_smoothing)
blended_pred_log =
    (1 - entity_weight) * model_pred_log
    + entity_weight * fallback_entity_price_log
```

The default `entity_smoothing` is `20`. That means 20 historical observations gives the entity prior 50% weight. This is a tunable hyperparameter, not a fixed truth.

The justification is that exact product variants are often price-stable. The criticism is that this is partly redundant with model features and should be validated or tuned.


## Anchor Calibration

Calibration uses the same-day anchor prices. For anchor rows:

```text
residual_log = log1p(anchor_price) - predicted_log_price
```

The median residual is used as a day-level correction. The pipeline also estimates residuals by `cat_id`, `shopId`, `brand`, and `promotionId`, with shrinkage toward the global day residual:

```text
weight = anchor_count / (anchor_count + calibration_smoothing)
```

The default `calibration_smoothing` is `8`. This keeps small anchor groups from overcorrecting the rest of the day.


## Anchor-Set Utilization Variants

The current recommended strategy is intentionally simple: use recent same-entity prices first, then fall back to the trained model only when recent entity history is missing.

1. **Last-price baseline**
   Use the recent entity prior directly, without CatBoost. This tests the anchor EDA finding that same-`modelId` last price is extremely strong.

2. **Hybrid last-price + fallback model**
   `hybrid_last_price_entity` is the primary candidate. It uses recent entity price when available, otherwise falls back to the entity-blend model.

3. **Hybrid last-price + calibrated fallback**
   `hybrid_last_price_calibrated_fallback` is the main anchor-use alternative. It keeps strong last-price rows untouched and applies anchor calibration only to fallback rows.

4. **Global / segment calibration**
   Estimate anchor residuals and apply a day-level plus segment-level correction. These variants remain useful as baselines for checking whether calibration is actually helping.

5. **Model selection using anchor error**
   Score candidate variants on anchor rows, choose the lowest-anchor-MAE variant for that day, and use it for hidden rows. This appears as `anchor_model_selected_*`.

`second_stage_residual` is sidelined for now. The code still supports it as an explicit experimental variant, but the normal validation comparison does not include it.


## Validation Methodology

Validation simulates the actual outage:

1. Hold out recent historical days.
2. Train only on earlier history.
3. Sample anchors from each held-out day.
4. Hide the remaining prices.
5. Run the same prediction and calibration logic.
6. Evaluate hidden rows using MAE, RMSE, and MAPE in raw price space.

The model predicts log price internally, but metrics are computed after converting predictions back with `expm1`.

The validation script also writes `validation_segments.csv`, which breaks metrics down by date, price bucket, history-count bucket, and calibration status. This is the main diagnostic for cases where calibration improves RMSE but hurts MAPE.

Safer calibration controls are available:

```bash
.venv/bin/python scripts/run_validation.py \
  --calibration-delta-cap 0.05 \
  --selective-calibration \
  --calibration-min-anchors 50 \
  --calibration-max-residual-iqr 0.10 \
  --calibration-min-abs-global-delta 0.01
```


In [ ]:
# Run outage-style validation from inside the notebook.
# The default comparison emphasizes the current best strategies:
# - hybrid_last_price_entity
# - hybrid_last_price_calibrated_fallback
# - calibration/model baselines for context
#
# Set INCLUDE_EXPERIMENTAL_VARIANTS = True only when you want to rerun sidelined experiments
# such as second_stage_residual.
# Set VALIDATION_ITERATIONS lower, e.g. 20, for a quick notebook smoke run.
# The reload is intentional: it prevents stale notebook imports after editing src/validation.py.
import importlib
import src.validation as validation_module

validation_module = importlib.reload(validation_module)
run_outage_validation = validation_module.run_outage_validation

RUN_VALIDATION = True
INCLUDE_EXPERIMENTAL_VARIANTS = False
VALIDATION_ITERATIONS = config.catboost_iterations
VALIDATION_OUTPUT_DIR = config.output_dir

if RUN_VALIDATION:
    validation_config = replace(
        config,
        output_dir=VALIDATION_OUTPUT_DIR,
        catboost_iterations=VALIDATION_ITERATIONS,
        include_experimental_variants=INCLUDE_EXPERIMENTAL_VARIANTS,
    )
    validation_results, validation_summary, validation_segments = run_outage_validation(
        train_raw,
        validation_config,
    )

    validation_config.output_dir.mkdir(exist_ok=True)
    validation_results.to_csv(validation_config.output_dir / "validation_results.csv", index=False)
    validation_summary.to_csv(validation_config.output_dir / "validation_summary.csv")
    validation_segments.to_csv(validation_config.output_dir / "validation_segments.csv", index=False)

    display(validation_results)
    display(validation_summary)

    anchor_variants = [
        "last_price_baseline",
        "hybrid_last_price_entity",
        "hybrid_last_price_calibrated_fallback",
        "global_calibrated",
        "entity_blend_calibrated",
    ]
    if INCLUDE_EXPERIMENTAL_VARIANTS:
        anchor_variants.append("second_stage_residual")

    anchor_comparison = validation_summary.reset_index().query(
        "base_model in @anchor_variants or base_model.str.startswith('anchor_model_selected')",
        engine="python",
    )
    display(anchor_comparison.sort_values("MAE"))

    segment_variants = anchor_variants + ["anchor_model_selected_hybrid_last_price_entity"]
    display(
        validation_segments
        .query("variant in @segment_variants or variant.str.startswith('anchor_model_selected')", engine="python")
        .sort_values(["segment", "segment_value", "variant"])
        .head(80)
    )
else:
    print("Validation skipped. Set RUN_VALIDATION = True to run it.")


In [ ]:
# Load previously saved validation outputs without rerunning training.
validation_results_path = config.output_dir / "validation_results.csv"
validation_summary_path = config.output_dir / "validation_summary.csv"
validation_segments_path = config.output_dir / "validation_segments.csv"

if validation_results_path.exists():
    saved_validation_results = pd.read_csv(validation_results_path)
    display(saved_validation_results)
else:
    print(f"Missing {validation_results_path}. Run the validation cell above first.")

if validation_summary_path.exists():
    saved_validation_summary = pd.read_csv(validation_summary_path)
    display(saved_validation_summary)

    saved_anchor_variants = [
        "last_price_baseline",
        "hybrid_last_price_entity",
        "hybrid_last_price_calibrated_fallback",
        "global_calibrated",
        "entity_blend_calibrated",
    ]
    saved_anchor_comparison = saved_validation_summary.query(
        "base_model in @saved_anchor_variants or base_model.str.startswith('anchor_model_selected')",
        engine="python",
    )
    display(saved_anchor_comparison.sort_values("MAE"))
else:
    print(f"Missing {validation_summary_path}. Run the validation cell above first.")

if validation_segments_path.exists():
    saved_validation_segments = pd.read_csv(validation_segments_path)
    saved_segment_variants = saved_anchor_variants + ["anchor_model_selected_hybrid_last_price_entity"]
    display(
        saved_validation_segments
        .query("variant in @saved_segment_variants or variant.str.startswith('anchor_model_selected')", engine="python")
        .sort_values(["segment", "segment_value", "variant"])
        .head(80)
    )
else:
    print(f"Missing {validation_segments_path}. Run the validation cell above first.")


## Hyperparameter Tuning

The pipeline includes a tuning script that searches both model hyperparameters and post-processing constants:

- CatBoost `learning_rate`
- CatBoost `depth`
- CatBoost `iterations`
- `entity_smoothing` for the entity prior blend
- `calibration_smoothing` for anchor group shrinkage
This is important because the original `20` and `8` smoothing constants were reasonable defaults, not learned values. Calibration caps and non-MAE selection metrics are available in the CLI as experimental options, but they are disabled in the default workflow for now.

Example smoke command:

```bash
.venv/bin/python scripts/tune_hyperparameters.py \
  --sample-rows 5000 \
  --validation-days 1 \
  --anchors-per-day 20 \
  --learning-rates 0.05 \
  --depths 6 \
  --iterations 5 \
  --entity-smoothing 10,20 \
  --calibration-smoothing 4,8 \
  --output-dir outputs/smoke_tuning
```


In [20]:
tuning_summary_path = config.output_dir / "tuning" / "tuning_summary.csv"
tuning_details_path = config.output_dir / "tuning" / "tuning_details.csv"

if tuning_summary_path.exists():
    tuning_summary = pd.read_csv(tuning_summary_path)
    display(tuning_summary.head(20))
else:
    print(f"Missing {tuning_summary_path}. Run scripts/tune_hyperparameters.py first.")

if tuning_details_path.exists():
    tuning_details = pd.read_csv(tuning_details_path)
    print("Detailed tuning rows:", len(tuning_details))
else:
    print(f"Missing {tuning_details_path}. Run scripts/tune_hyperparameters.py first.")


Missing outputs/tuning/tuning_summary.csv. Run scripts/tune_hyperparameters.py first.
Missing outputs/tuning/tuning_details.csv. Run scripts/tune_hyperparameters.py first.


## Prediction Output

The prediction script trains on the full training file, predicts the test file, fills only missing prices, and verifies that known anchor prices are preserved.


In [21]:
completed_path = config.output_dir / "completed_test_predictions.csv"

if completed_path.exists():
    completed = pd.read_csv(completed_path)
    print("Rows:", len(completed))
    print("Missing prices:", completed[config.target].isna().sum())
    display(completed.head())
else:
    print(f"Missing {completed_path}. Run scripts/predict_test.py first.")


Rows: 25900
Missing prices: 0


,capturedAt,shopId,itemId,modelId,price,priceBeforeDiscount,promotionId,cat_id,stock,normal_stock,...,review_rating,total_rating_count,cmt_count,shop_rating,shop_response_rate,shop_follower_count,is_official_shop,is_verified,is_preferred_plus_seller,date
0,2025-03-22 04:27:05.325,1009757562,27904817781,139002028392,7533911,NaN,missing,missing,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,2025-03-22
1,2025-03-22 04:27:05.325,1009757562,27904817781,139002028392,7533911,NaN,missing,missing,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,2025-03-22
2,2025-03-22 04:27:05.325,1009757562,27904817781,241162534603,8494802,NaN,missing,missing,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,2025-03-22
3,2025-03-22 04:27:05.325,1009757562,27904817781,241162534603,8494802,NaN,missing,missing,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,2025-03-22
4,2025-03-22 04:27:05.325,1009757562,27904817781,241162534604,7533911,NaN,missing,missing,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,2025-03-22


In [22]:
# Optional: run prediction from inside the notebook.
# This can take time with the default 1200 CatBoost iterations.
RUN_FULL_PREDICTION = False

if RUN_FULL_PREDICTION:
    submission = run_prediction(config)
    display(submission.head())
